In [2]:
import os, random, librosa, numpy as np, warnings
import matplotlib.cm as cm
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings('ignore')

SR_FIXED      = 16000
TARGET_SEC    = 1.0
TARGET_SAMP   = int(TARGET_SEC * SR_FIXED)

output_dir  = "/kaggle/working/spectrograms"
dataset_dir = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm"
categories  = ['real', 'fake']
split_map   = {'training': 'train', 'validation': 'val', 'testing': 'test'}

for split in ['train', 'val', 'test']:
    for cat in categories:
        os.makedirs(f"{output_dir}/{split}/{cat}", exist_ok=True)

def convert(audio_path, save_path, mode='train'):

    try:
        y, _ = librosa.load(audio_path, sr=SR_FIXED)

        if len(y) < TARGET_SAMP:
            return "skipped"

        if mode == 'train':
            start = random.randint(0, len(y) - TARGET_SAMP)
        else:
            start = (len(y) - TARGET_SAMP) // 2

        y_crop = y[start : start + TARGET_SAMP]

        S    = librosa.feature.melspectrogram(y=y_crop, sr=SR_FIXED, n_mels=128, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)

        S_norm  = (S_dB - S_dB.min()) / (S_dB.max() - S_dB.min() + 1e-8)
        S_norm  = np.flipud(S_norm)
        colored = (cm.magma(S_norm)[:, :, :3] * 255).astype(np.uint8)
        Image.fromarray(colored).resize((128, 128), Image.LANCZOS).save(save_path)
        return "success"

    except Exception as e:
        print(f"\nError: {audio_path} — {e}")
        return "error"

print(f"Generating | sr={SR_FIXED} | target={TARGET_SEC}s | NO PADDING | random crop (train)\n")

counts = {}
for split_in, split_out in split_map.items():
    mode = 'train' if split_in == 'training' else 'eval'

    for cat in categories:
        in_folder  = f"{dataset_dir}/{split_in}/{cat}"
        out_folder = f"{output_dir}/{split_out}/{cat}"

        if not os.path.exists(in_folder):
            print(f"Missing: {in_folder}")
            continue

        files   = [f for f in os.listdir(in_folder) if f.endswith('.wav')]
        pbar    = tqdm(files, desc=f"{split_in}/{cat}")
        skipped = 0

        for f in pbar:
            out_path = f"{out_folder}/{f.replace('.wav', '.png')}"
            if not os.path.exists(out_path):
                r = convert(f"{in_folder}/{f}", out_path, mode=mode)
                if r == "skipped":
                    skipped += 1
                    pbar.set_postfix({'skipped <1s': skipped})

        saved = len(os.listdir(out_folder))
        counts[f"{split_out}/{cat}"] = saved
        print(f"  → {saved} saved, {skipped} skipped (under 1s)\n")

print("\nFinal counts:")
for k, v in counts.items():
    print(f"  {k}: {v}")

Generating | sr=16000 | target=1.0s | NO PADDING | random crop (train)



training/real: 100%|██████████| 26941/26941 [12:15<00:00, 36.63it/s, skipped <1s=156]


  → 26785 saved, 156 skipped (under 1s)



training/fake: 100%|██████████| 26927/26927 [08:53<00:00, 50.48it/s, skipped <1s=3936]


  → 22991 saved, 3936 skipped (under 1s)



...

  → 4583 saved, 815 skipped (under 1s)



testing/real: 100%|██████████| 2264/2264 [00:46<00:00, 48.94it/s, skipped <1s=21]


  → 2243 saved, 21 skipped (under 1s)



testing/fake: 100%|██████████| 2370/2370 [00:43<00:00, 54.54it/s, skipped <1s=266]

  → 2104 saved, 266 skipped (under 1s)


Final counts:
  train/real: 26785
  train/fake: 22991
  val/real: 5365
  val/fake: 4583
  test/real: 2243
  test/fake: 2104


In [3]:
for split in ['train', 'val', 'test']:
    r = len(os.listdir(f'/kaggle/working/spectrograms/{split}/real'))
    f = len(os.listdir(f'/kaggle/working/spectrograms/{split}/fake'))
    ratio = r/f if f > 0 else 0
    print(f"{split}: real={r}, fake={f}, ratio={ratio:.2f}")

train: real=26785, fake=22991, ratio=1.17
val: real=5365, fake=4583, ratio=1.17
test: real=2243, fake=2104, ratio=1.07


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve
from tqdm import tqdm
import numpy as np
import time
import torch.nn.functional as F
import warnings

warnings.filterwarnings('ignore')

DATA_BASE_DIR = '/kaggle/working/spectrograms'

n_real = len(os.listdir(os.path.join(DATA_BASE_DIR, 'train/real')))
n_fake = len(os.listdir(os.path.join(DATA_BASE_DIR, 'train/fake')))
total  = n_real + n_fake

w_fake = total / (2.0 * n_fake)
w_real = total / (2.0 * n_real)
print(f"real={n_real}, fake={n_fake} | w_fake={w_fake:.3f}, w_real={w_real:.3f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.4),     
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_BASE_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_BASE_DIR, 'val'),   transform=eval_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_BASE_DIR, 'test'),  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nClass mapping : {train_dataset.class_to_idx}")   # Should be: fake=0, real=1
print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Test samples  : {len(test_dataset)}")

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

model = DeepfakeDetector().to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel ready — {trainable:,} / {total:,} params trainable")

def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return eer * 100, thresholds[idx]

def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_labels, all_scores = [], []

    with torch.no_grad():
        for images, labels in loader:
            scores = model(images.to(device)).cpu().numpy().flatten()
            all_scores.extend(scores)
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    all_preds  = (all_scores >= threshold).astype(int)

    acc = accuracy_score(all_labels, all_preds) * 100
    f1  = f1_score(all_labels, all_preds, zero_division=0) * 100
    eer, eer_thresh = compute_eer(all_labels, all_scores)

    return acc, f1, eer, eer_thresh, all_labels, all_preds, all_scores

def weighted_bce(outputs, labels_t):
    """BCELoss with per-class weights to handle slight imbalance."""
    w = torch.where(labels_t == 0,
                    torch.tensor(w_fake, device=device, dtype=torch.float32),
                    torch.tensor(w_real, device=device, dtype=torch.float32))
    return F.binary_cross_entropy(outputs, labels_t, weight=w)

optimizer = optim.Adam([
    {'params': model.backbone.parameters(),   'lr': 5e-5},
    {'params': model.classifier.parameters(), 'lr': 5e-4}
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

EPOCHS       = 15
best_val_acc = 0.0

SAVE_PATH    = "/kaggle/working/best_model.pth"

print("\n Training started\n")
print(f"{'Ep':>3} | {'Train':>6} | {'Val Acc':>7} | {'Val F1':>6} | {'EER':>5} | {'LR':>7} | Time")
print("─" * 65)

for epoch in range(EPOCHS):
    model.train()
    correct, total_samples, loss_sum = 0, 0, 0.0
    start = time.time()

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS}", leave=False):
        images   = images.to(device)
        labels_t = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = weighted_bce(outputs, labels_t)
        loss.backward()
        optimizer.step()

        loss_sum       += loss.item()
        correct        += ((outputs >= 0.5).float() == labels_t).sum().item()
        total_samples  += labels_t.size(0)

    train_acc = (correct / total_samples) * 100
    val_acc, val_f1, val_eer, _, _, _, _ = evaluate(model, val_loader)
    scheduler.step(val_acc)

    lr      = optimizer.param_groups[0]['lr']
    elapsed = time.time() - start
    marker  = " " if val_acc > best_val_acc else ""

    print(f"{epoch+1:>3} | {train_acc:>5.1f}% | {val_acc:>6.1f}%  | {val_f1:>5.1f}% | "
          f"{val_eer:>4.1f}% | {lr:.0e} | {elapsed:.0f}s{marker}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)

print(f"\n Training done. Best val accuracy: {best_val_acc:.1f}%")

print("\nFinding optimal threshold on validation set...")
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
_, _, _, _, val_true, _, val_scores = evaluate(model, val_loader)

best_thresh, best_f1_val = 0.5, 0.0
for t in np.arange(0.15, 0.85, 0.01):
    preds = (val_scores >= t).astype(int)
    f1    = f1_score(val_true, preds, zero_division=0)
    if f1 > best_f1_val:
        best_f1_val, best_thresh = f1, t

print(f"Optimal threshold : {best_thresh:.2f}")
print(f"Val F1 at threshold: {best_f1_val*100:.1f}%")

test_acc, test_f1, test_eer, _, y_true, y_pred, _ = evaluate(
    model, test_loader, threshold=best_thresh
)

print("\n" + "=" * 52)
print("           FINAL TEST RESULTS")
print("=" * 52)
print(f"  Accuracy  : {test_acc:.2f}%     (target ≥ 80%)")
print(f"  F1 Score  : {test_f1:.2f}%     (target ≥ 80%)")
print(f"  EER       : {test_eer:.2f}%     (target ≤ 12%)")
print(f"  Threshold : {best_thresh:.2f}")
print()
print("  Confusion Matrix:")
print("                 Pred Fake   Pred Real")
cm = confusion_matrix(y_true, y_pred)
print(f"  True Fake  :  {cm[0][0]:>6}      {cm[0][1]:>6}")
print(f"  True Real  :  {cm[1][0]:>6}      {cm[1][1]:>6}")
per_class_fake = cm[0][0] / (cm[0][0] + cm[0][1]) * 100
per_class_real = cm[1][1] / (cm[1][0] + cm[1][1]) * 100
print(f"\n  Per-class recall → Fake: {per_class_fake:.1f}%  |  Real: {per_class_real:.1f}%")
print("=" * 52)

torch.save({
    'model_state_dict' : model.state_dict(),
    'class_to_idx'     : train_dataset.class_to_idx,
    'threshold'        : float(best_thresh),
}, '/kaggle/working/final_model.pth')

print("\n final_model.pth saved to Kaggle Working Directory — ready for Streamlit.")


real=26785, fake=22991 | w_fake=1.083, w_real=0.929
Training on: cuda

Class mapping : {'fake': 0, 'real': 1}
Train samples : 49776
Val samples   : 9948
Test samples  : 4347
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s] 



Model ready — 11,308,609 / 11,308,609 params trainable

🚀 Training started

 Ep |  Train | Val Acc | Val F1 |   EER |      LR | Time
─────────────────────────────────────────────────────────────────


  1 |  98.8% |   99.8%  |  99.8% |  0.2% | 5e-05 | 72s ✅


  2 |  99.8% |   99.6%  |  99.7% |  0.2% | 5e-05 | 73s


...

 11 | 100.0% |   99.9%  |  99.9% |  0.0% | 3e-05 | 72s


 12 | 100.0% |  100.0%  | 100.0% |  0.0% | 1e-05 | 72s


 13 | 100.0% |  100.0%  | 100.0% |  0.0% | 1e-05 | 72s


 14 | 100.0% |  100.0%  | 100.0% |  0.0% | 1e-05 | 72s


 15 | 100.0% |  100.0%  | 100.0% |  0.0% | 1e-05 | 73s

🎉 Training done. Best val accuracy: 100.0%

Finding optimal threshold on validation set...
Optimal threshold : 0.27
Val F1 at threshold: 100.0%

           FINAL TEST RESULTS
  Accuracy  : 51.67%     (target ≥ 80%)
  F1 Score  : 68.10%     (target ≥ 80%)
  EER       : 17.44%     (target ≤ 12%)
  Threshold : 0.27

  Confusion Matrix:
                 Pred Fake   Pred Real
  True Fake  :       3        2101
  True Real  :       0        2243

  Per-class recall → Fake: 0.1%  |  Real: 100.0%

✅ final_model.pth saved to Kaggle Working Directory — ready for Streamlit.
